# 03 — Mutual information: estimator, batch stratification, and redundancy

**Claims under test**
1. The stratified (batch-conditioned) MI estimator yields one number per gene, and agrees with the
   pooled estimate (mouse 0.927, human 0.816) while *strengthening* the cross-species result.
2. MI is not redundant with |Spearman| — MI-unique genes are measurably more nonmonotone.

**Known caveat carried in the code:** the stratified script applies normalization to already-normalized
data. Harmless in principle (MI is invariant under monotone transforms), and the pooled/stratified
agreement is consistent with that - but note the agreement is only moderate in human (0.816, or
0.727 on the matched universe), so it is weaker empirical support than the mouse value alone
suggests. Remove the redundant normalization before this becomes the paper's code.

In [1]:
# --- paths & helpers (edit ROOT if you move things) ---------------------------
import os, json, glob, pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

X    = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
REV  = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions")
NET  = pathlib.Path("/n/netscratch/eddy_lab/Everyone/nhilgert")
GP   = REV/"crossspecies_radial_gp"
DELIV= X/"data/deliverables"
ISO  = X/"results/isodepth"
MIS  = X/"results/mi_stratified"
DATA = pathlib.Path("data")     # staged artifacts, see README
ORTH = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/human_ivd_repro/orthogroups/ensembl_human_mouse_components.tsv")

HS_DELIV = DELIV/"2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MM_DELIV = DELIV/"2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"   # NOTE: "XMens", not "Mouse"

_checks = []
def verify(label, got, expected, tol=0.02, note=""):
    """Recompute vs recorded. Prints PASS/MISMATCH and accumulates a summary."""
    try:
        ok = (abs(float(got) - float(expected)) <= tol * max(1.0, abs(float(expected)))
              if expected is not None else None)
    except (TypeError, ValueError):
        ok = (got == expected)
    tag = "PASS " if ok else ("MISMATCH" if ok is False else "n/a  ")
    _checks.append((label, got, expected, tag))
    print(f"[{tag}] {label}: recomputed={got}  recorded={expected}" + (f"   ({note})" if note else ""))
    return ok

def check_summary():
    d = pd.DataFrame(_checks, columns=["claim","recomputed","recorded","status"])
    n_bad = int((d.status.str.strip()=="MISMATCH").sum())
    print(f"\n{len(d)} checks | {n_bad} MISMATCH")
    return d

def ortholog_1to1():
    o = pd.read_csv(ORTH, sep="\t")
    o = o[o.homology_type=="ortholog_one2one"]
    piv = o.pivot_table(index="component_id", columns="species", values="gene_symbol",
                        aggfunc=lambda s: s.iloc[0] if len(s)==1 else np.nan).dropna()
    return piv[["human","mouse"]].drop_duplicates()

print("data dir:", DATA.resolve(), "| files:", len(list(DATA.glob("*"))) if DATA.exists() else "MISSING")
print("env ok |", "deliverables:", HS_DELIV.exists(), MM_DELIV.exists(), "| isodepth dir:", ISO.exists())


data dir: /n/lab_storage/eddy_lab/users/nhilgert/xmens_final/notebooks/verify/data | files: 29
env ok | deliverables: True True | isodepth dir: True


## 3.1 The estimator — recompute MI for a handful of genes from scratch

This is the actual computation, not a table lookup: sklearn `mutual_info_regression` with a permutation null.

In [2]:
import scanpy as sc
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import spearmanr

GENES = ["Runx1","Cebpb","Gja1","Cd63","Ebf1","Meis2","Dcn","Vcan"]
a = sc.read_h5ad(ISO/"mouse_fibdec_isodepth.h5ad")
iso = a.obs["isodepth_consensus"].astype(float)
keep = iso.notna().to_numpy()
rng = np.random.default_rng(0)
sub = rng.choice(np.where(keep)[0], min(3000, keep.sum()), replace=False)   # subsample: MI cost is O(n log n)
y = iso.to_numpy()[sub]
rows = []
for g in GENES:
    if g not in a.var_names: continue
    v = a[sub, g].X
    v = np.asarray(v.todense()).ravel() if hasattr(v, "todense") else np.asarray(v).ravel()
    mi = float(mutual_info_regression(v.reshape(-1,1), y, random_state=0)[0])
    null = [float(mutual_info_regression(v.reshape(-1,1), rng.permutation(y), random_state=i)[0]) for i in range(20)]
    rows.append(dict(gene=g, mi=round(mi,4), null_mean=round(float(np.mean(null)),4),
                     z=round((mi-np.mean(null))/max(np.std(null),1e-9),1),
                     rho=round(float(spearmanr(v, y).statistic),3)))
R = pd.DataFrame(rows); print(R.to_string(index=False))
print("\nNote sign: MI is unsigned; rho carries direction. Genes with high MI and near-zero rho are")
print("the nonmonotone cases MI exists to find.")
del a

 gene     mi  null_mean    z    rho
Runx1 0.1686     0.0063 21.8  0.497
Cebpb 0.0974     0.0023 29.3  0.395
 Gja1 0.1505     0.0067 17.6  0.401
 Cd63 0.0802     0.0040 13.6  0.338
 Ebf1 0.2023     0.0037 40.5 -0.404
Meis2 0.1244     0.0047 15.4 -0.355
  Dcn 0.0540     0.0025 14.3 -0.156
 Vcan 0.0548     0.0039  9.2 -0.089

Note sign: MI is unsigned; rho carries direction. Genes with high MI and near-zero rho are
the nonmonotone cases MI exists to find.


## 3.2 Stratified vs pooled — one number per gene, not one list per sample

Recorded: **mouse 0.927, human 0.816** (0.727 on the matched 9,550-gene universe) - the two
species are NOT the same; mouse strata = 7 samples, human = 6.
**Uninduced is not among the mouse strata** — the MI tables never used it.

In [3]:
from scipy.stats import spearmanr
for sp in ("mouse","human"):
    f = MIS/f"{sp}_isodepth_stratified.csv"
    if not f.exists(): print("missing", f); continue
    T = pd.read_csv(f)
    per = [c for c in T.columns if c.startswith("mi_") and c not in ("mi_stratified","mi_pooled")]
    r = float(spearmanr(T.mi_stratified, T.mi_pooled).statistic)
    print(f"--- {sp}: {len(T):,} genes | {len(per)} strata")
    print("    strata:", [c.replace("mi_","") for c in per])
    print(f"    Spearman(stratified, pooled) = {r:.4f}")
    expected_agree = {"mouse": 0.927, "human": 0.816}[sp]
    verify(f"{sp} stratified-vs-pooled agreement", round(r,3), expected_agree, tol=0.05)
    verify(f"{sp} n strata", len(per), 7 if sp=="mouse" else 6, tol=0)
    if sp=="mouse":
        verify("uninduced EXCLUDED from mouse MI strata",
               bool(not any("uninduced" in c for c in per)), True,
               note="=> imputed uninduced isodepth never entered the conservation analysis")

--- mouse: 14,095 genes | 7 strata
    strata: ['1_dpi_early_mck1', '1_dpi_late_mck12a', '1_dpi_late_mck12b', '2_dpi_mck15', '3_dpi_mck10a', '3_dpi_mck10b', '4_dpi_bleeding_mck11a']
    Spearman(stratified, pooled) = 0.9273
[PASS ] mouse stratified-vs-pooled agreement: recomputed=0.927  recorded=0.927
[PASS ] mouse n strata: recomputed=7  recorded=7
[PASS ] uninduced EXCLUDED from mouse MI strata: recomputed=True  recorded=True   (=> imputed uninduced isodepth never entered the conservation analysis)


--- human: 15,011 genes | 6 strata
    strata: ['menstrual_mck5a', 'menstrual_mck5b', 'menstrual_mck5c', 'secretory_mck6a', 'secretory_mck6b', 'secretory_mck6c']
    Spearman(stratified, pooled) = 0.8157
[PASS ] human stratified-vs-pooled agreement: recomputed=0.816  recorded=0.816
[PASS ] human n strata: recomputed=6  recorded=6


## 3.3 Is MI redundant with correlation?

Recorded: rho(MI, |Spearman|) = 0.433 mouse / 0.469 human; top-decile overlap 46.8% / 56.0%;
MI-unique genes ~1.5x more curved (0.0578 vs 0.0303 mouse); conserved-set Jaccard 0.223.

In [4]:
p_ = DATA/"mi_vs_rho_redundancy.json"
if not p_.exists(): p_ = None
if p_ and p_.exists():
    d = json.load(open(p_))
    for sp in ("mouse","human"):
        v = d[sp]
        print(f"--- {sp}: rho(MI,|rho|)={v['rho_MI_vs_absrho']}  top10 overlap={v['frac_overlap']:.3f}"
              f"  MI-only n={v['n_mi_only']}")
        print(f"    curvature: MI-only {v['mi_only_median_abs_quad']:.4f} vs shared {v['shared_median_abs_quad']:.4f}")
        verify(f"{sp} rho(MI, |Spearman|)", v["rho_MI_vs_absrho"], 0.433 if sp=="mouse" else 0.469)
        verify(f"{sp} MI-unique genes more curved",
               bool(v["mi_only_median_abs_quad"] > v["shared_median_abs_quad"]), True)
    verify("conserved-set Jaccard (MI vs rho)", d["conserved_jaccard"], 0.223)
else:
    print("mi_vs_rho_redundancy.json not found in ~ - point p= at it")

--- mouse: rho(MI,|rho|)=0.433  top10 overlap=0.468  MI-only n=651
    curvature: MI-only 0.0578 vs shared 0.0303
[PASS ] mouse rho(MI, |Spearman|): recomputed=0.433  recorded=0.433
[PASS ] mouse MI-unique genes more curved: recomputed=True  recorded=True
--- human: rho(MI,|rho|)=0.469  top10 overlap=0.560  MI-only n=554
    curvature: MI-only 0.0462 vs shared 0.0347
[PASS ] human rho(MI, |Spearman|): recomputed=0.469  recorded=0.469
[PASS ] human MI-unique genes more curved: recomputed=True  recorded=True
[PASS ] conserved-set Jaccard (MI vs rho): recomputed=0.223  recorded=0.223


## Summary

In [5]:
check_summary()


10 checks | 0 MISMATCH


,claim,recomputed,recorded,status
0,mouse stratified-vs-pooled agreement,0.927,0.927,PASS
1,mouse n strata,7,7,PASS
2,uninduced EXCLUDED from mouse MI strata,True,True,PASS
3,human stratified-vs-pooled agreement,0.816,0.816,PASS
4,human n strata,6,6,PASS
5,"mouse rho(MI, |Spearman|)",0.433,0.433,PASS
6,mouse MI-unique genes more curved,True,True,PASS
7,"human rho(MI, |Spearman|)",0.469,0.469,PASS
8,human MI-unique genes more curved,True,True,PASS
9,conserved-set Jaccard (MI vs rho),0.223,0.223,PASS
